# KIKA: MCNP PERT Sensitivity Analysis Report Pipeline

This notebook demonstrates the complete pipeline for sensitivity analysis and uncertainty quantification using MCNP PERT cards:

1. **Load MCNP PERT results** → `SensitivityData`
2. **Generate comprehensive report** → HTML with sensitivity plots and second-order analysis
3. **Export to SDF format** → For comparison and downstream UQ
4. **Run uncertainty quantification** → Sandwich formula with covariance matrices

The new `MCNPPertReport` class provides a standardized way to analyze MCNP PERT results with minimal code.

# KIKA: MCNP PERT Sensitivity Analysis Report Pipeline

This notebook demonstrates the complete pipeline for sensitivity analysis and uncertainty quantification using MCNP PERT cards:

1. **Load MCNP PERT results** → `SensitivityData`
2. **Generate comprehensive report** → HTML with sensitivity plots and second-order analysis
3. **Export to SDF format** → For comparison and downstream UQ
4. **Run uncertainty quantification** → Sandwich formula with covariance matrices

## Architecture Overview

```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MCNP Input +   │     │ SensitivityData  │     │    SDFData      │
│  MCTAL Output   │ ──► │   (Full data)    │ ──► │ (Common format) │
└─────────────────┘     └──────────────────┘     └─────────────────┘
                               │                        │
                               ▼                        ▼
                        ┌──────────────┐         ┌──────────────┐
                        │ MCNPPertReport│         │  UQReport    │
                        │ (Full report) │         │ (Sandwich)   │
                        └──────────────┘         └──────────────┘
```

In [ ]:
import kika
from kika.sensitivities import (
    compute_sensitivity,
    create_sdf_data,
    MCNPPertReport,
    UQReport,
    ComparisonReport
)
from pathlib import Path
import pandas as pd

# Setup paths
repo_root = Path.cwd().resolve().parent
data_dir = repo_root / 'examples' / 'data'
output_dir = Path.cwd() / 'output'
output_dir.mkdir(exist_ok=True)

print(f"Data directory: {data_dir}")
print(f"Output directory: {output_dir}")

## Step 1: Load MCNP PERT Results

The `compute_sensitivity` function reads PERT cards from MCNP input and results from MCTAL files to calculate sensitivity coefficients. It supports:
- First-order sensitivities (always)
- Second-order Taylor coefficients (if PERT method=3 was used)

In [ ]:
# Load example files
inputfile = data_dir / 'inputfile_example_1.i'
mctalfile = data_dir / 'mctalfile_example_1.m'

# Compute sensitivity for Fe-56
sens_fe56 = compute_sensitivity(
    inputfile=inputfile,
    mctalfile=mctalfile,
    tally=4,
    zaid=26056,  # Fe-56
    label="Fe-56 Sensitivity"
)

# Display the sensitivity data object
sens_fe56

In [ ]:
# You can also compute sensitivity for multiple nuclides
# (if PERT cards exist for other nuclides in the input file)

# For this demo, we'll use just Fe-56
all_sensitivities = [sens_fe56]

# Quick summary of available data
print(f"Nuclide: {sens_fe56.nuclide}")
print(f"Reactions available: {sens_fe56.reactions}")
print(f"Energy bins: {sens_fe56.energies}")
print(f"Has second-order data: {bool(sens_fe56.coefficients)}")

## Step 2: Generate MCNP PERT Report

The `MCNPPertReport` class generates comprehensive reports from MCNP PERT sensitivity data:
- Summary tables
- Sensitivity coefficient tables (sortable by magnitude)
- Sensitivity profile plots
- Second-order analysis (if available): nonlinearity ratios, perturbed response comparisons

In [ ]:
# Create the report
report = MCNPPertReport(
    sensitivity_data=all_sensitivities,
    title="MCNP PERT Sensitivity Analysis - Demo"
)

# View summary
report.summary()

In [ ]:
# View sensitivity table for integral energy
sens_table = report.sensitivity_table(energy='integral')
sens_table.head(15)

In [ ]:
# Plot top sensitivities
fig = report.plot_sensitivities(energy='integral', top_n=10)
fig

In [ ]:
# If second-order data is available, we can analyze nonlinearity
if report.has_second_order:
    print("Second-order analysis available!")
    
    # Get available energies with second-order data
    for sd in report.data:
        if sd.coefficients:
            print(f"\n{sd.nuclide} has second-order data for energies:")
            for energy in list(sd.coefficients.keys())[:3]:
                reactions = list(sd.coefficients[energy].keys())
                print(f"  {energy}: MT {reactions}")
else:
    print("No second-order data available (PERT method=2 only)")

In [ ]:
# Plot second-order analysis if available
if report.has_second_order:
    # Plot for a specific energy and reaction
    energy_key = sens_fe56.energies[2] if len(sens_fe56.energies) > 2 else 'integral'
    fig = report.plot_second_order_analysis(energy=energy_key, nuclide_idx=0)
    if fig:
        fig

### Save Complete HTML Report

Generate a standalone HTML report with all analysis embedded.

In [ ]:
# Save HTML report
report_path = output_dir / 'mcnp_pert_report.html'
report.save_html(
    filepath=str(report_path),
    energy='integral',
    include_plots=True,
    include_second_order=True
)

print(f"\nReport saved! Open in browser: {report_path}")

## Step 3: Export to SDF Format

SDF (Sensitivity Data Format) is a common interchange format that allows:
- Comparison between different codes (MCNP, Serpent, SCALE, etc.)
- Uncertainty quantification using the sandwich formula
- Storage and sharing of sensitivity results

In [ ]:
# Convert to SDF format
sdf_data = report.to_sdf(
    energy='integral',
    title='MCNP_PERT_Demo'
)

# Display SDF data structure
sdf_data

In [ ]:
# Save SDF file
sdf_data.write_file(output_dir=str(output_dir))

## Step 4: Uncertainty Quantification

The `UQReport` class uses the sandwich formula σ²_R = S^T Σ S to propagate nuclear data uncertainties from covariance matrices to the response.

**Note**: This requires covariance matrices (e.g., from ENDF/B-VIII.0 via NJOY processing). For this demo, we'll show the interface but skip actual computation if no covariance data is available.

In [ ]:
# UQ requires covariance matrices
# Here's how you would set it up:

# from kika.cov import CovMat
# 
# # Load covariance matrix for Fe-56
# cov_fe56 = CovMat.from_file('path/to/fe56_covmat.dat')
# 
# # Create UQ report
# uq_report = UQReport(
#     sdf_data=sdf_data,
#     cov_mat=cov_fe56,
#     title='MCNP PERT Uncertainty Quantification'
# )
# 
# # Run calculation and view results
# uq_report.compute()
# uq_report.summary()

print("UQ workflow requires covariance matrices.")
print("See documentation for loading CovMat from NJOY output.")

### Example UQ Output Structure

When covariance matrices are available, `UQReport` provides:

```python
uq_report.summary()          # Summary DataFrame
uq_report.contribution_table()  # Per-reaction breakdown
uq_report.plot_contributions()  # Bar chart of contributors
uq_report.plot_pie()         # Pie chart of contributions
uq_report.save_html(path)    # Complete HTML report
```

The results include:
- Total uncertainty (combined statistical + nuclear data)
- Per-reaction variance contributions
- Cross-correlation effects

## Summary

This notebook demonstrated the complete MCNP PERT sensitivity analysis pipeline:

| Step | Function/Class | Output |
|------|---------------|--------|
| 1. Load PERT results | `compute_sensitivity()` | `SensitivityData` |
| 2. Generate report | `MCNPPertReport` | HTML with plots, tables |
| 3. Export to SDF | `report.to_sdf()` | `SDFData` → .sdf file |
| 4. Run UQ | `UQReport` | Uncertainty breakdown |

### Key Features

**SensitivityData** (MCNP-specific):
- First-order sensitivity coefficients
- Second-order Taylor coefficients (if method=3)
- Detector energy dependence
- Nonlinearity analysis (c2/c1 ratios)

**SDFData** (Common format):
- Integrated sensitivities
- Compatible with SCALE, Serpent, other codes
- Input for sandwich UQ

In [ ]:
# Final summary of outputs
print("Generated outputs:")
for f in output_dir.glob('*'):
    print(f"  - {f.name}")